# Lab 02 · Analyze without changing

Qiskit Compiler Contribution Lab · Learner notebook  
**Material version:** pilot-revision-1-2026-10-22 · revised after first-round feedback; learner retest pending.

**Build:** a compiler analysis pass that publishes a circuit inventory.  
**Practice:** separating observations from rewrites, defining exactly what a count means, and testing preservation.  
**Prerequisites:** basic Python, circuit literacy, and [Lab 01](01-circuit-dag.ipynb).

This lesson is general compiler practice; it does not depend on the author's pull requests. It is an independent learning project, not an official Qiskit tutorial.

Run cells in order. The starter should finish with **NOT YET**, not a kernel exception. Edit the solution cell, rerun it, and rerun the checker. No quantum hardware or account is needed.

## 1. Check your environment

From the folder containing `pyproject.toml` and `uv.lock`:

```sh
uv sync --locked
uv run qclab doctor
uv run python -c "import sys; print(sys.executable)"
uv run jupyter lab notebooks/02-analysis-pass.ipynb
```

Compare the first cell's interpreter path with the terminal's project `.venv`. The kernel's “Python 3” label alone is not sufficient. In VS Code, select the environment using the notebook's **Select Kernel** control. Follow the [setup checklist](../docs/setup.md) if imports fail, especially when working in WSL.

This notebook needs the installed `qclab` helper, not just a copied notebook file. Restart an already-open kernel after syncing updated dependencies.

In [ ]:
import sys

print("Notebook kernel:", sys.executable)
print("Compare its .venv folder with the terminal command in the setup checklist.")

In [ ]:
import sys

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import Gate
from qiskit.converters import circuit_to_dag
from qiskit.transpiler import AnalysisPass, PassManager

from qclab.notebook import check_solution

print("Python:", sys.version.split()[0], "| Qiskit:", qiskit.__version__)
assert qiskit.__version__ == "2.6.2", "Select this project's .venv kernel."

## 2. A pass that observes

A compiler pass is a scheduled unit of work on an intermediate representation. In Qiskit, an `AnalysisPass` inspects a DAG and publishes information in `self.property_set`; a `TransformationPass` changes the DAG.

A `PassManager` runs passes on a circuit and exposes their shared properties. The return from `manager.run(circuit)` is a **circuit**, not our report.

First, a complete small example. Predict whether the idle third qubit will be included.

In [ ]:
class CountWires(AnalysisPass):
    def run(self, dag):
        self.property_set["wire_count"] = dag.num_qubits()


small = QuantumCircuit(3)
small.h(0)
wire_manager = PassManager(CountWires())
unchanged = wire_manager.run(small)
print("Reported qubits:", wire_manager.property_set["wire_count"])
print("Circuit unchanged:", unchanged == small)
assert wire_manager.property_set["wire_count"] == 3
assert unchanged == small

## 3. Inspect before implementing

The next circuit has an idle qubit and an idle classical bit. Before running it, predict the number of operations and the number of two-qubit **gates**. Why might those counts differ from the number of two-wire operations?

In [ ]:
circuit = QuantumCircuit(3, 2, name="inventory_example")
circuit.h(0)
circuit.h(1)
circuit.cx(0, 1)
circuit.barrier(0, 1)
circuit.measure(1, 0)
circuit.reset(1)
circuit.delay(10, 0)
print(circuit.draw(output="text"))

In [ ]:
dag = circuit_to_dag(circuit)
for node in dag.op_nodes():
    print(
        node.name,
        "| qubits:",
        len(node.qargs),
        "| classical bits:",
        len(node.cargs),
        "| Gate:",
        isinstance(node.op, Gate),
    )

### Worked boundary: two wires do not necessarily mean a gate

An **operation node** wraps an operation plus its qubit and classical-bit arguments. A `Gate` is a kind of quantum operation; measurements, resets, and barriers are not gates. The name of an operation is not a reliable type check.

| Example | In operation counts? | In two-qubit gate count? |
|---|---|---|
| Two-wire barrier | Yes | **No**—it is not a `Gate` |
| CX or custom two-qubit `Gate` | Yes | Yes |
| Measurement or reset | Yes | No |
| Three-qubit `Gate` | Yes | No—wrong arity |

Before running the next cell, predict both classifications. The custom gate is deliberately not called `cx`. We inspect it without simulating it or expanding a definition.

In [ ]:
boundary_circuit = QuantumCircuit(2)
boundary_circuit.barrier(0, 1)
boundary_circuit.append(Gate("custom_pair", 2, []), [0, 1])
boundary_dag = circuit_to_dag(boundary_circuit)

pair_classification = {}
for node in boundary_dag.op_nodes():
    is_pair_gate = isinstance(node.op, Gate) and len(node.qargs) == 2
    pair_classification[node.name] = is_pair_gate
    print(node.name, "| two wires:", len(node.qargs) == 2, "| two-qubit gate:", is_pair_gate)

assert pair_classification == {"barrier": False, "custom_pair": True}

### Worked boundary: top-level only means no recursion

**This report inventories the current DAG's top-level nodes. Do not descend into branches or expand gate definitions.**

A control-flow operation is a container for blocks. The next circuit has one top-level `if_else` with ten CX operations inside its true branch. Our report counts **one `if_else` and zero top-level two-qubit gates**. The nested CX operations still exist; they are simply outside this report's scope.

The inventory does not tell you whether the branch executes. A composite two-qubit gate is different: it is itself a `Gate`, so it counts once regardless of its internal definition. Neither count estimates native hardware cost.

In [ ]:
branch_circuit = QuantumCircuit(2, 1)
with branch_circuit.if_test((branch_circuit.clbits[0], True)):
    for _ in range(10):
        branch_circuit.cx(0, 1)

branch_dag = circuit_to_dag(branch_circuit)
top_level_nodes = list(branch_dag.op_nodes())
print("Top-level operation names:", [node.name for node in top_level_nodes])
print("Operations in the true branch:", len(top_level_nodes[0].op.blocks[0].data))
assert [node.name for node in top_level_nodes] == ["if_else"]
assert len(top_level_nodes[0].op.blocks[0].data) == 10
assert not isinstance(top_level_nodes[0].op, Gate)
print("Expected report: operation_counts={'if_else': 1}, num_two_qubit_gates=0")

## 4. Your task: CircuitReport

Implement a no-argument `CircuitReport(AnalysisPass)`. Its `run(dag)` must assign a fresh dictionary to `self.property_set["circuit_report"]`:

| Field | Meaning |
|---|---|
| `num_qubits` | All qubit wires, including idle ones |
| `num_clbits` | All classical-bit wires, including idle ones |
| `operation_counts` | Dictionary from operation name to number of top-level occurrences; include gates, measurements, resets, barriers, delays, and control-flow operations |
| `num_two_qubit_gates` | Number of top-level nodes whose operation is a `Gate` (including subclasses) and has exactly two qubit arguments |

Count occurrences, not distinct names. An empty DAG still produces all four fields, with zero counts and an empty operation dictionary. A two-wire barrier is **not** a gate. A custom two-qubit gate **is** a gate: do not hard-code names.

**Top-level only: do not expand composite gate definitions or recurse into control-flow blocks.** An `if_else` node contributes one entry to `operation_counts`, regardless of its branches. This is a **structural, top-level inventory**, not a count of dynamically executed gates or hardware cost.

Preserve the input DAG, operations, parameters, wires, name, metadata, and global phase. Preserve unrelated property-set entries. Return `None`. Reusing the pass or pass manager must not accumulate counts or mutate a previously published report.

For the example above, expect:
```python
{
    "num_qubits": 3,
    "num_clbits": 2,
    "operation_counts": {
        "h": 2, "cx": 1, "barrier": 1, "measure": 1, "reset": 1, "delay": 1
    },
    "num_two_qubit_gates": 1,
}
```
Dictionary ordering does not matter.

In [ ]:
class CircuitReport(AnalysisPass):
    """Complete the report; count top-level operations, not nested definitions."""

    def run(self, dag):
        # Wire counts work, but both operation summaries are unfinished.
        self.property_set["circuit_report"] = {
            "num_qubits": dag.num_qubits(),
            "num_clbits": dag.num_clbits(),
            "operation_counts": {},
            "num_two_qubit_gates": 0,
        }

## 5. Run your pass — create a new instance after an edit

Initially the wire counts work and the operation summaries are unfinished. After changing the solution:

1. **Rerun the `class CircuitReport` definition cell.** Editing text alone changes nothing in the running kernel.
2. **Rerun the next cell.** It constructs a new `CircuitReport()` and a new `PassManager`, then runs the circuit.
3. **Rerun the checker cell.** It snapshots the current class, not the old manager's instance.
4. When you reach the downstream pipeline, rerun that cell too; it also creates a new instance.

A class definition is a blueprint; `CircuitReport()` creates an instance. An instance already stored in a manager does not acquire a newly defined class. If results seem unchanged after an edit, follow the sequence above before debugging the algorithm.

Within each `run`, allocate fresh counters and a new report dictionary. Reusing a pass must not accumulate results or change a report that a caller kept from a previous run.

In [ ]:
manager = PassManager(CircuitReport())
output_circuit = manager.run(circuit)
print(manager.property_set["circuit_report"])
print("Circuit unchanged:", output_circuit == circuit)

## 6. Run the shared checks

Pass the **class**, not an instance: `check_solution(CircuitReport, lab="02")`.

The checker snapshots the current notebook-defined class and runs the 14 public acceptance cases in a fresh Python process. It returns feedback without stopping the notebook for an unfinished solution. It does not overwrite the terminal exercise.

The checker executes your code locally; it is not a security sandbox. Only run code you trust.

In [ ]:
result = check_solution(CircuitReport, lab="02")
print("Acceptance status:", "PASS" if result.passed else "NOT YET")
if not result.passed:
    print("Next action: read the first named failure below; check one report field at a time.")
    print("Rerun the class definition and create a new manager before inspecting your report.")
print(result.output)

## Hints — open one at a time

<details><summary>Hint 1: Where do observations go?</summary>

Implement <code>run(self, dag)</code>. Read <code>dag.op_nodes()</code>, then publish the dictionary through <code>self.property_set["circuit_report"]</code>. Do not return a replacement DAG.

</details>

<details><summary>Hint 2: What is an occurrence?</summary>

Each node contributes one count under its name, even if another node has the same name. Begin with a fresh dictionary and counter on every run.

</details>

<details><summary>Hint 3: What qualifies as a two-qubit gate?</summary>

Check both the operation's type with <code>isinstance(node.op, Gate)</code> and its qubit arguments. A barrier can span two wires without being a gate. A custom gate need not be called <code>cx</code>.

</details>

<details><summary>Hint 4: Why avoid mutation?</summary>

Read nodes; don't remove, replace, or reorder them. Allocate a new report rather than updating a dictionary that another pass may already hold. Keep unrelated property-set entries intact.

</details>

## 7. Let another pass use the report

A report becomes useful when later passes can consume it. This example records a simple warning flag; it does not change the circuit or estimate device performance. Once your solution is correct, the flag for our example should be `True`.

In [ ]:
class FlagPairGates(AnalysisPass):
    def run(self, dag):
        report = self.property_set["circuit_report"]
        self.property_set["has_pair_gates"] = report["num_two_qubit_gates"] > 0


pipeline = PassManager([CircuitReport(), FlagPairGates()])
pipeline.run(circuit)
print("Contains top-level two-qubit gates:", pipeline.property_set["has_pair_gates"])

## 8. Add your own regression

Choose a boundary: repeated gates, an idle wire, a custom two-qubit gate, or a two-wire barrier. Write a tiny circuit and an assertion with a hand-worked expected value.

Then temporarily introduce a mistake in your solution. Does your test fail for the intended reason? Restore the solution and rerun the checks. Do not change the acceptance tests to force a pass.

In [ ]:
# Your circuit, pass manager, and assertion here.

## 9. Explain the contribution

In your own words:

1. Why is `operation_counts` not a gate count? Why is `num_two_qubit_gates` not necessarily a native hardware gate count?
2. What does the report say about a circuit with one `if_else` containing ten CX gates? What does it **not** tell you?
3. Why must the report be fresh on each run?
4. How would you detect an analysis pass that quietly deletes a barrier?
5. Which test did you write, and which plausible mistake does it catch?

**Done means:** shared checks pass, your regression catches a mistake, you can explain the scope, and **Restart Kernel → Run All** succeeds without hidden cell state.

Only then compare with the [instructor notebook](instructor/02-analysis-pass-solution.ipynb). Lab 03 is planned in the [roadmap](../docs/roadmap.md). Facilitators can use the [observation guide](../docs/facilitator-guide.md). Passing the checker is not the same as a successful clean-kernel notebook run; record both.

### Official API references

- [AnalysisPass](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.AnalysisPass)
- [PassManager](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.PassManager)
- [DAGCircuit](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.dagcircuit.DAGCircuit)

This repository pins Qiskit 2.6.2; the linked “latest” documentation may change.